#  InfoNCE Loss + CLIPModel Integration


---
##  Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/prajwl7676/mini-clip-DLAI.git

%cd /content/mini-clip-DLAI
!pip install -q -r requirements.txt



In [ ]:
import sys
sys.path.insert(0, '/content/mini-clip-DLAI')
!pwd
!ls -l

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from transformers import DistilBertTokenizer
from torch.utils.data import DataLoader

from src.model_architecture import CLIPModel
from src.dataset import FlickrDataset, get_transform

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
##  Implement InfoNCE loss


In [ ]:
def clip_loss(image_embeddings, text_embeddings, temperature):

    B = image_embeddings.shape[0]

    # Step 1 — N×N cosine similarity matrix
    # S[i,j] = dot product of image_i and text_j (cosine sim since L2-normed)
    S = image_embeddings @ text_embeddings.T          # (B, B)

    # Step 2 — scale by temperature
    logits = S / temperature                          # (B, B)

    # Step 3 — labels: image i is matched with text i
    labels = torch.arange(B, device=image_embeddings.device)

    # Step 4 — symmetric cross-entropy
    # Row  i: which column j maximises logits[i,j]? Answer should be j=i
    # Col  j: which row    i maximises logits[i,j]? Answer should be i=j
    loss_i2t = F.cross_entropy(logits,   labels)      # image → text
    loss_t2i = F.cross_entropy(logits.T, labels)      # text  → image

    return (loss_i2t + loss_t2i) / 2.0


print('clip_loss defined.')

---
## Sanity check: what does the loss value mean?

For a batch of N pairs, a random model produces loss ≈ log(N).  
This is the baseline — the model is guessing randomly.  
After training, the loss should be well below log(N).

In [ ]:
import math

for B in [8, 32, 64, 128]:
    expected_random_loss = math.log(B)
    print(f'Batch size {B:4d} → random loss baseline ≈ log({B}) = {expected_random_loss:.3f}')

print()
print('If your trained model achieves loss << log(B), it has learned something.')
print('If loss ≈ log(B), the model is still random — something is wrong.')

---
## Test loss on fake embeddings

Three scenarios:  
1. **Random embeddings** → loss should be ≈ log(B)  
2. **Perfect embeddings** (diagonal = 1, off-diagonal = 0) → loss should be ≈ 0  
3. **Swapped embeddings** (diagonal = 0) → loss should be very high

In [ ]:
B, D = 8, 256
tau  = torch.tensor(0.07)

# Scenario 1: random embeddings
img_rand = F.normalize(torch.randn(B, D), dim=-1)
txt_rand = F.normalize(torch.randn(B, D), dim=-1)
loss_random = clip_loss(img_rand, txt_rand, tau)

# Scenario 2: perfect embeddings — image i == text i
perfect = F.normalize(torch.randn(B, D), dim=-1)
loss_perfect = clip_loss(perfect, perfect.clone(), tau)

# Scenario 3: worst case — image i matched with text (i+1)
txt_wrong = torch.roll(perfect, shifts=1, dims=0)   # shift captions by 1
loss_worst = clip_loss(perfect, txt_wrong, tau)

print(f'Random embeddings loss : {loss_random.item():.4f}  (expected ≈ {math.log(B):.3f})')
print(f'Perfect match loss     : {loss_perfect.item():.4f}  (expected ≈ 0.0)')
print(f'Worst case loss        : {loss_worst.item():.4f}  (expected >> {math.log(B):.3f})')

assert loss_perfect.item() < 0.01,   'Perfect loss should be near 0'
assert loss_random.item()  > 1.0,    'Random loss should be > 1'
assert loss_worst.item()   > loss_random.item(), 'Worst case > random'
print('\nAll loss sanity checks passed.')

---
## Effect of temperature on loss


In [ ]:
img = F.normalize(torch.randn(16, 256), dim=-1)
txt = F.normalize(torch.randn(16, 256), dim=-1)

temps  = [0.01, 0.05, 0.07, 0.1, 0.3, 0.5, 1.0]
losses = [clip_loss(img, txt, torch.tensor(t)).item() for t in temps]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temps, losses, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axhline(math.log(16), color='crimson', linestyle='--', linewidth=1.2, label=f'log(16) = {math.log(16):.2f}')
ax.set_xlabel('Temperature τ')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Effect of temperature on contrastive loss (random embeddings, B=16)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('assets/temperature_effect.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/temperature_effect.png')

---
## Load real data for a complete integration test

In [ ]:
dataset_hf = load_dataset('nlphuji/flickr30k', split='test[:500]')
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

ds     = FlickrDataset(dataset_hf, tokenizer, get_transform(train=True))
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=2,
                    pin_memory=True, drop_last=True)

images, tokens = next(iter(loader))
images = images.to(device)
input_ids  = tokens['input_ids'].to(device)
attn_mask  = tokens['attention_mask'].to(device)

print(f'Batch ready — images: {images.shape}, tokens: {input_ids.shape}')

---
## Full integration: model → loss → backward

In [ ]:
model     = CLIPModel(embedding_dim=256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# One training step
model.train()

# Step 1: forward pass
img_emb, txt_emb, tau = model(images, input_ids, attn_mask)

# Step 2: compute loss
loss = clip_loss(img_emb, txt_emb, tau)

# Step 3: backward pass
optimizer.zero_grad()
loss.backward()

# Step 4: gradient clipping (prevents exploding gradients)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Step 5: update weights
optimizer.step()

print(f'loss after step 1 : {loss.item():.4f}')
print(f'temperature τ     : {tau.item():.4f}')
print(f'log(32) baseline  : {math.log(32):.4f}')
print()
print('One complete training step succeeded.')

---
## Mini training loop: 10 steps to confirm loss decreases

In [ ]:
model     = CLIPModel(embedding_dim=256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

loss_history = []

# Run 10 steps on the same batch (overfit test)
# Loss should decrease — proves the gradients are flowing correctly
for step in range(10):
    model.train()
    img_emb, txt_emb, tau = model(images, input_ids, attn_mask)
    loss = clip_loss(img_emb, txt_emb, tau)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_history.append(loss.item())
    print(f'Step {step+1:2d} | loss: {loss.item():.4f} | τ: {tau.item():.4f}')

print()
if loss_history[-1] < loss_history[0]:
    print('Loss is decreasing — gradients are flowing correctly.')
else:
    print('WARNING: loss did not decrease — check model and loss function.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 11), loss_history, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axhline(math.log(32), color='crimson', linestyle='--', linewidth=1.2,
           label=f'Random baseline log(32)={math.log(32):.2f}')
ax.set_xlabel('Training step')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Loss over 10 steps on one batch (overfit test)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('assets/overfit_test_loss.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Visualise similarity matrix AFTER 10 steps

The diagonal should already be slightly brighter after just 10 steps.

In [ ]:
model.eval()
with torch.no_grad():
    img_emb, txt_emb, tau = model(images[:8], input_ids[:8], attn_mask[:8])
    sim = (img_emb @ txt_emb.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels([f'T{i}' for i in range(8)], fontsize=9)
ax.set_yticklabels([f'I{i}' for i in range(8)], fontsize=9)
ax.set_title('Similarity matrix — AFTER 10 steps\n(diagonal = matched pairs)')

for i in range(8):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False,
                                edgecolor='navy', lw=2))
    ax.text(i, i, f'{sim[i,i]:.2f}', ha='center', va='center',
            fontsize=9, fontweight='bold', color='navy')

plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.savefig('assets/similarity_matrix_after_10_steps.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Confirm src/loss.py import

In [ ]:
from src.loss import clip_loss as clip_loss_src, similarity_matrix

# Test imported loss matches inline version
img_t = F.normalize(torch.randn(8, 256), dim=-1)
txt_t = F.normalize(torch.randn(8, 256), dim=-1)
tau_t = torch.tensor(0.07)

loss_inline = clip_loss(img_t, txt_t, tau_t)
loss_src    = clip_loss_src(img_t, txt_t, tau_t)

assert torch.allclose(loss_inline, loss_src), 'Mismatch between inline and src loss!'

# Test similarity_matrix utility
sim = similarity_matrix(img_t, txt_t)
assert sim.shape == (8, 8), 'Wrong similarity matrix shape!'

print('src.loss import works correctly.')
print(f'  clip_loss output : {loss_src.item():.4f}')
print(f'  similarity_matrix shape : {sim.shape}')